# 03 AI Communicator

Reads relevant markets and creates immutable JSON index cards using a placeholder ScenarioAnalysisGPT wrapper. Mock mode is enabled by default until the real API contract is configured.


## Setup


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "falnama_config.yaml").exists() or (candidate / "polymarket_geopolitics_anomaly_detection_pilot.ipynb").exists():
            return candidate
    raise RuntimeError("Could not locate Falnama project root. Run from inside the Falnama folder.")

PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config" / "falnama_config.yaml"
RUN_TIME_UTC = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def load_config() -> dict[str, Any]:
    with CONFIG_PATH.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

CONFIG = load_config()
REPOSITORIES = CONFIG.get("repositories", {})
for repo_rel in REPOSITORIES.values():
    (PROJECT_ROOT / repo_rel).mkdir(parents=True, exist_ok=True)

RUN_LOG_DIR = PROJECT_ROOT / REPOSITORIES.get("run_logs", "repositories/run_logs")
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)


def repo_path(key: str, default: str) -> Path:
    path = PROJECT_ROOT / REPOSITORIES.get(key, default)
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_run_log(notebook_name: str, records: list[dict[str, Any]]) -> Path:
    path = RUN_LOG_DIR / f"{notebook_name}_{RUN_TIME_UTC.replace(':', '').replace('-', '')}.jsonl"
    with path.open("x", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps({"run_time_utc": RUN_TIME_UTC, **record}, default=str) + "\n")
    return path


def load_csv_nonempty(path: Path) -> pd.DataFrame | None:
    if not path.exists() or path.stat().st_size <= 1:
        return None
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return None
    return df if not df.empty else None


def normalize_timestamp(value: Any) -> pd.Timestamp:
    if value is None or value == "" or (isinstance(value, float) and np.isnan(value)):
        return pd.NaT
    if isinstance(value, pd.Timestamp):
        return value.tz_localize("UTC") if value.tzinfo is None else value.tz_convert("UTC")
    if isinstance(value, (int, float, np.integer, np.floating)):
        unit = "ms" if float(value) > 10_000_000_000 else "s"
        return pd.to_datetime(value, unit=unit, utc=True, errors="coerce")
    return pd.to_datetime(value, utc=True, errors="coerce")


def parse_jsonish(value: Any, default: Any = None) -> Any:
    if default is None:
        default = []
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return default
        try:
            return json.loads(text)
        except Exception:
            return value
    return value


def first_present(obj: dict[str, Any] | pd.Series, keys: list[str], default: Any = None) -> Any:
    for key in keys:
        if key in obj and obj[key] not in (None, "") and not (isinstance(obj[key], float) and np.isnan(obj[key])):
            return obj[key]
    return default


def safe_slug(value: Any, fallback: str = "unknown") -> str:
    text = str(value if value not in (None, "") else fallback).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text[:100] or fallback


## Immutable Index Card Generation
The wrapper has explicit TODOs for endpoint, auth, payload, and response parsing. Every card is schema-validated, content-hashed, and written with exclusive-create semantics.


In [2]:
from jsonschema import Draft202012Validator

RELEVANT_DIR = repo_path("relevant_markets", "repositories/relevant_markets")
INDEX_CARD_DIR = repo_path("index_cards", "repositories/index_cards")
SCHEMA_PATH = PROJECT_ROOT / "schemas" / "index_card_schema.json"
LOGS: list[dict[str, Any]] = []

with SCHEMA_PATH.open("r", encoding="utf-8") as f:
    INDEX_CARD_SCHEMA = json.load(f)
INDEX_CARD_VALIDATOR = Draft202012Validator(INDEX_CARD_SCHEMA)


def newest_file(directory: Path, pattern: str) -> Path | None:
    files = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None


def load_relevant_markets() -> tuple[pd.DataFrame, str | None]:
    path = newest_file(RELEVANT_DIR, "relevant_markets_*.csv")
    if path is not None:
        df = load_csv_nonempty(path)
        if df is not None:
            return df, str(path)
    legacy_ranked = load_csv_nonempty(PROJECT_ROOT / CONFIG.get("legacy_inputs", {}).get("legacy_ranked_anomalies", ""))
    if legacy_ranked is not None:
        legacy_ranked = legacy_ranked.rename(columns={"question": "market_name", "market_url": "market_url"})
        if "market_slug" not in legacy_ranked.columns:
            legacy_ranked["market_slug"] = legacy_ranked.get("market_name", pd.Series(dtype=str)).map(lambda x: safe_slug(x, "market"))
        return legacy_ranked, str(PROJECT_ROOT / CONFIG.get("legacy_inputs", {}).get("legacy_ranked_anomalies", ""))
    mock = pd.DataFrame([{"market_id": "mock-market-001", "market_slug": "mock-geopolitical-risk-market", "market_name": "Mock geopolitical escalation market for smoke testing", "market_url": None}])
    return mock, None


def market_context_from_row(row: pd.Series, source_file: str | None) -> dict[str, Any]:
    name = first_present(row, ["market_name", "question", "title"], "Unknown market")
    slug = first_present(row, ["market_slug", "slug"], safe_slug(name, "market"))
    return {
        "market_id": None if pd.isna(first_present(row, ["market_id", "id"], None)) else str(first_present(row, ["market_id", "id"], None)),
        "market_slug": None if pd.isna(slug) else str(slug),
        "market_name": str(name),
        "market_url": None if pd.isna(first_present(row, ["market_url", "url"], None)) else first_present(row, ["market_url", "url"], None),
        "source_market_file": source_file,
        "raw_market_record": {k: (None if pd.isna(v) else v) for k, v in row.to_dict().items()},
    }


def choose_mock_prediction(market_context: dict[str, Any], config: dict[str, Any]) -> dict[str, Any]:
    name = market_context.get("market_name", "").lower()
    if "thai" in name or "thailand" in name:
        asset, ticker, instrument, asset_class = "Thailand equities", "THD", "iShares MSCI Thailand ETF", "etf"
    elif "oil" in name or "iran" in name or "middle east" in name:
        asset, ticker, instrument, asset_class = "Crude oil", "USO", "United States Oil Fund", "commodity"
    elif "china" in name or "taiwan" in name:
        asset, ticker, instrument, asset_class = "China equity risk proxy", "FXI", "iShares China Large-Cap ETF", "etf"
    else:
        asset, ticker, instrument, asset_class = "Emerging-market risk proxy", "EEM", "iShares MSCI Emerging Markets ETF", "etf"
    min_move = float(config.get("minimum_expected_move_bps", 700))
    return {
        "asset": asset,
        "ticker": ticker,
        "tradable_instrument_name": instrument,
        "asset_class": asset_class,
        "expected_direction": "down",
        "expected_return_12h_bps": -max(min_move + 150, 850),
        "confidence": 0.62,
        "confidence_interval_bps": [-1400, -250],
        "trade_eligibility": {"eligible": True, "reason": "Mock output meets configured minimum expected move for pipeline testing.", "minimum_expected_move_bps": min_move},
        "reasoning": "MOCK: maps prediction-market surprise to a broad liquid market proxy. Replace with ScenarioAnalysisGPT reasoning before research use.",
        "time_plan": {
            "30m": "Check whether the market anomaly persists and whether public news confirms it.",
            "1h": "Compare proxy movement with regional risk assets.",
            "2h": "Reassess whether the thesis is stale or contradicted.",
            "6h": "Monitor liquidity and any official statements.",
            "12h": "Close the paper research window and archive results."
        }
    }


def call_scenario_analysis_gpt(market_context: dict[str, Any], config: dict[str, Any]) -> dict[str, Any]:
    """Placeholder wrapper for ScenarioAnalysisGPT.

    TODO: Replace PLACEHOLDER endpoint with the production ScenarioAnalysisGPT URL.
    TODO: Load auth from config['scenario_analysis_api_key_env_var'] and attach it according to the real auth contract.
    TODO: Finalize request payload fields once the API contract is known.
    TODO: Parse the production response into the immutable index-card schema below.
    """
    if config.get("scenario_analysis_mock_mode", True) or config.get("scenario_analysis_api_endpoint") == "PLACEHOLDER":
        return {"mode": "mock", "predictions": [choose_mock_prediction(market_context, config)]}
    raise NotImplementedError("ScenarioAnalysisGPT production API contract is not configured. Keep mock mode enabled until it is.")


def canonical_hash(card: dict[str, Any]) -> str:
    payload = json.loads(json.dumps(card, default=str))
    payload["card_hash"] = ""
    blob = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


def build_index_card(market_context: dict[str, Any], config: dict[str, Any]) -> dict[str, Any]:
    created_time = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
    response = call_scenario_analysis_gpt(market_context, config)
    source = {
        "market_id": market_context.get("market_id"),
        "market_slug": market_context.get("market_slug"),
        "market_name": market_context.get("market_name", "Unknown market"),
        "source_market_file": market_context.get("source_market_file"),
        "market_url": market_context.get("market_url"),
    }
    natural_key = source.get("market_id") or source.get("market_slug") or safe_slug(source.get("market_name"), "market")
    card = {
        "card_id": f"{natural_key}_{created_time}",
        "card_version": "v1-mock" if response.get("mode") == "mock" else "v1",
        "card_hash": "",
        "do_not_revise": True,
        "created_time_utc": created_time,
        "source": source,
        "market_name": source["market_name"],
        "realized_outcome": "Unknown",
        "conclusion_time_utc": None,
        "prediction_window": "12h",
        "predictions": response.get("predictions", []),
        "evidence_used": [
            "prediction market anomaly context",
            "historical event-study analogues",
            "public news confirmation",
            "asset sensitivity mapping"
        ],
        "uncertainty_notes": ["MOCK MODE output. Do not treat as an investment recommendation." if response.get("mode") == "mock" else "ScenarioAnalysisGPT output requires downstream audit."],
        "llm_justification": {
            "summary": "Mock ScenarioAnalysisGPT card generated for pipeline testing." if response.get("mode") == "mock" else "ScenarioAnalysisGPT generated card.",
            "key_assumptions": ["Prediction-market anomaly is informative rather than noise.", "Selected asset proxy is sufficiently sensitive to the event."],
            "failure_modes": ["Market anomaly reverses.", "Public information is already fully priced.", "Proxy asset has weak event sensitivity."],
            "why_not_ex_post": "The card is timestamped, content-hashed, and never overwritten. Trading must reject cards created after a live trigger time."
        }
    }
    card["card_hash"] = canonical_hash(card)
    errors = sorted(INDEX_CARD_VALIDATOR.iter_errors(card), key=lambda e: e.path)
    if errors:
        raise ValueError("Index card schema validation failed: " + "; ".join(error.message for error in errors))
    return card


def write_immutable_card(card: dict[str, Any]) -> Path:
    natural_key = card["source"].get("market_id") or card["source"].get("market_slug") or safe_slug(card["market_name"], "market")
    stamp = card["created_time_utc"].replace(":", "").replace("-", "")
    path = INDEX_CARD_DIR / f"{safe_slug(natural_key)}_{stamp}_{card['card_hash'][:12]}.json"
    if path.exists():
        raise FileExistsError(f"Immutable index card path already exists: {path}")
    path.write_text(json.dumps(card, indent=2, ensure_ascii=False), encoding="utf-8")
    return path


## Run AI Communicator


In [3]:
markets_df, source_file = load_relevant_markets()
written_cards = []
errors = []
for _, row in markets_df.iterrows():
    try:
        context = market_context_from_row(row, source_file)
        card = build_index_card(context, CONFIG)
        path = write_immutable_card(card)
        written_cards.append({"card_id": card["card_id"], "card_hash": card["card_hash"], "path": str(path), "market_name": card["market_name"]})
    except Exception as exc:
        errors.append({"market_name": first_present(row, ["market_name", "question"], None), "error": repr(exc)})

log_path = write_run_log("03_ai_communicator", LOGS + [{"event": "ai_communicator_outputs", "source_file": source_file, "cards_written": len(written_cards), "errors": errors}])
print(f"Cards written: {len(written_cards):,}")
print(f"Errors: {len(errors):,}")
print(log_path)
display(pd.DataFrame(written_cards).head(20))
if errors:
    display(pd.DataFrame(errors))


Cards written: 21
Errors: 0
/Users/R2-D2/Documents/Codex/Falnama/repositories/run_logs/03_ai_communicator_20260609T004003Z.jsonl


,card_id,card_hash,path,market_name
0,923041_2026-06-09T02:15:21Z,b6e494f7a5e85472452be10bc7be53569874180f483b5f2a4861cef9bf6adc78,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/923041_20260609T021521Z_b6e494f7a5e8.json,Will People’s Party (PPLE) win the most seats in the 2026 Thai legislative election?
1,923042_2026-06-09T02:15:21Z,facac0802dc1fd88bb577827d764e374e9aa19634b145de08763c067975d86be,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/923042_20260609T021521Z_facac0802dc1.json,Will Bhumjaithai Party (BJT) win the most seats in the 2026 Thai legislative election?
2,1271753_2026-06-09T02:15:21Z,a8e17eb06faeac5984c6b11632b5c4691b265fa96020a20ffb84c0d9623c51ad,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/1271753_20260609T021521Z_a8e17eb06fae.json,Will Bhumjaithai Party (BJT) finish in second place by number of seats in the 2026 Thai legislative election?
3,1271752_2026-06-09T02:15:21Z,58dcd0630c1b84be7ce327dd17ea5d4f35ba17e1378ebb6231f88f2feea77d84,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/1271752_20260609T021521Z_58dcd0630c1b.json,Will People’s Party (PPLE) finish in second place by number of seats in the 2026 Thai legislative election?
4,1272508_2026-06-09T02:15:21Z,3f99eb21f50b69327687bf6c016f2845f8a52c2f5a4c981505ce14da83e0d1d3,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/1272508_20260609T021521Z_3f99eb21f50b.json,Will the People’s Party (PPLE) win between 120 and 134 seats in the 2026 Thai legislative election?
5,1272538_2026-06-09T02:15:21Z,e1ca12a0749940f111c74da23330c767a0fa968a3f2ede4c140fccf72e3c16e6,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/1272538_20260609T021521Z_e1ca12a07499.json,Will the Bhumjaithai Party (BJT) win 140 or more seats in the 2026 Thai legislative election?
6,1271758_2026-06-09T02:15:21Z,8aebc031c059fce8524ae886a83b201956dc11bf9b73e0b85afccedcc12bc6da,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/1271758_20260609T021521Z_8aebc031c059.json,Will Chart Thai Pattana Party (CTPP) finish in second place by number of seats in the 2026 Thai legislative election?
7,1271755_2026-06-09T02:15:21Z,e43823627ce2f9e73274a5056f338402ad1b34b3245f2e6444807535ce759702,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/1271755_20260609T021521Z_e43823627ce2.json,Will Palang Pracharath Party (PPRP) finish in second place by number of seats in the 2026 Thai legislative election?
8,1272537_2026-06-09T02:15:21Z,da861514f028e04216ddef06c0930709d9d5349fb0e0fc9a110ca21fc65496dc,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/1272537_20260609T021521Z_da861514f028.json,Will the Bhumjaithai Party (BJT) win between 130 and 139 seats in the 2026 Thai legislative election?
9,1271757_2026-06-09T02:15:21Z,448e0cf520edfc40a3b3bffcd7bdba488fc3367020a1074e667456e661e5f074,/Users/R2-D2/Documents/Codex/Falnama/repositories/index_cards/1271757_20260609T021521Z_448e0cf520ed.json,Will Democrat Party (DP) finish in second place by number of seats in the 2026 Thai legislative election?
